# ✈️ Naive PDF Reader by Collins Aerospace
### Text-to-Text Naive RAG using LangGraph + Groq


In [ ]:
# Install necessary libraries
!pip install -q langchain langgraph langchain-community langchain-core pypdf chromadb sentence-transformers groq gradio

In [ ]:
# Import libraries
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langgraph.graph import StateGraph
from typing import TypedDict
from groq import Groq
import gradio as gr

In [ ]:
# Set Groq API Key
os.environ['GROQ_API_KEY'] = 'YOUR_GROQ_API_KEY'
client = Groq()

In [ ]:
# LLM Call Function
def call_llm(prompt):
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama3-8b-8192"
    )
    return response.choices[0].message.content

In [ ]:
# Graph State
class GraphState(TypedDict):
    question: str
    context: str
    answer: str

In [ ]:
# Load & Index PDF
def load_and_index(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings()
    vectorstore = Chroma.from_documents(chunks, embeddings)

    return vectorstore.as_retriever()

In [ ]:
# Retrieve Step
def retrieve(state):
    docs = retriever.get_relevant_documents(state['question'])
    context = "\n".join([doc.page_content for doc in docs])
    return {"context": context}

In [ ]:
# Generate Step
def generate(state):
    prompt = f"""
    You are an aerospace AI expert.

    Context:
    {state['context']}

    Question:
    {state['question']}

    Provide structured output:
    - Summary
    - Risk Factors
    - Prevention Strategy
    - Final Recommendation
    """

    answer = call_llm(prompt)
    return {"answer": answer}

In [ ]:
# Build LangGraph
graph = StateGraph(GraphState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
app = graph.compile()

In [ ]:
# Evaluation Metrics
def evaluate_rag(question, answer, context):
    relevance = len(answer) / (len(context) + 1)
    faithfulness = "High" if answer in context else "Moderate"
    return {"Relevance": relevance, "Faithfulness": faithfulness}

In [ ]:
# Pipeline Function
def rag_pipeline(pdf, question):
    global retriever
    retriever = load_and_index(pdf.name)

    result = app.invoke({"question": question})

    metrics = evaluate_rag(question, result['answer'], result.get('context', ''))

    return f"""
    ✨ Answer:
    {result['answer']}

    📊 Metrics:
    Relevance: {metrics['Relevance']}
    Faithfulness: {metrics['Faithfulness']}
    """

In [ ]:
# Gradio UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# ✈️ Naive PDF Reader by Collins Aerospace")

    gr.HTML("""
    <div style='overflow:hidden;'>
      <div style='animation: marquee 10s linear infinite;'>🚀 Prevent Catastrophic Failure During Launch</div>
    </div>
    <style>
    @keyframes marquee {0%{transform:translateX(100%);}100%{transform:translateX(-100%);}}
    </style>
    """)

    pdf_input = gr.File(label="Upload PDF")
    question = gr.Textbox(label="Ask Question")
    output = gr.Textbox(lines=15)

    btn = gr.Button("Generate")
    btn.click(rag_pipeline, inputs=[pdf_input, question], outputs=output)

demo.launch()